# Background Uncertainties — Solutions

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy  as np
import tables as tb
import pandas as pd
import matplotlib.pyplot as plt

import scipy.stats     as stats  # statistics and Many PDFs 
#import scipy.optimize  as optimize # Minimice funtions

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, sys
from pathlib import Path

# Find the project root: honours FANAL_ROOT env-var, otherwise walks up from cwd
_env = os.environ.get('FANAL_ROOT') or os.environ.get('USCFANALDIR')
if _env and Path(_env, 'data').is_dir():
    rootpath = str(Path(_env).resolve())
else:
    rootpath = str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'data').is_dir() and (p / 'ana').is_dir()))
if rootpath not in sys.path:
    sys.path.insert(0, rootpath)
print('Fanal root : ', rootpath)

In [ ]:
import core.pltext  as pltext   # extensions for plotting histograms
import core.hfit    as hfit     # extension to fit histograms
import core.efit    as efit     # Fit Utilites - Includes Extend Likelihood Fit with composite PDFs
import core.utils   as ut       # generic utilities
import ana.fanal    as fn       # analysis functions specific to fanal
import ana.pltfanal as pltfn    # plotting fanal functions
import     collpars as collpars # collaboration specific parameters
pltext.style()

In [ ]:
collaboration = collpars.collaboration
sel_erange    = collpars.sel_erange
sel_eroi      = collpars.sel_eroi

eff_Bi_blind  = collpars.eff_Bi_blind
eff_Tl_blind  = collpars.eff_Tl_blind

eff_Bi_E      = collpars.eff_Bi_E
eff_Tl_E      = collpars.eff_Tl_E
eff_Bi_RoI    = collpars.eff_Bi_RoI
eff_Tl_RoI    = collpars.eff_Tl_RoI

print('Collaboration         : {:s}'.format(collaboration))
print('Energy range          : ({:6.3f}, {:6.3f}) MeV'.format(*sel_erange))
print('RoI    range          : ({:6.3f}, {:6.3f}) MeV'.format(*sel_eroi))
print('Bi eff in E range     : {:6.4f} %'.format(100.*eff_Bi_E))
print('Tl eff in E range     : {:6.4f} %'.format(100.*eff_Tl_E))
print('Bi eff in RoI         : {:6.4f} %'.format(100.*eff_Bi_RoI))
print('Tl eff in RoI         : {:6.4f} %'.format(100.*eff_Tl_RoI))

In [ ]:
# number of  blind events
n_Bi_blind = eff_Bi_blind * collpars.n_Bi_total
n_Tl_blind = eff_Tl_blind * collpars.n_Tl_total
n_blind    = [n_Bi_blind, n_Tl_blind]
print('Number of bkg events in blind data : Bi = {:6.2f}, Tl = {:6.2f}.'.format(*n_blind))

In [ ]:
filename = '/data/fanal_' + collaboration + '.h5'
print('Data path and filename : ', rootpath + filename)

mcbi       = pd.read_hdf(rootpath + filename, key = 'mc/bi214')   # MC Bi
mctl       = pd.read_hdf(rootpath + filename, key = 'mc/tl208')   # MC Tl
data_blind = pd.read_hdf(rootpath + filename, key = 'data/blind') # blind data

mc_samples         = [mcbi, mctl]
sample_names       = ['Bi', 'Tl']
sample_names_latex = [r'$^{214}$Bi', r'$^{208}$Tl']

In [ ]:
# get the blind mc samples
mcs_blind  = [mc[fn.selection_blind(mc)] for mc in mc_samples]

varname   = 'E'
refnames  = (varname,) 
refranges = (sel_erange,)

fit        = fn.prepare_fit_ell(mcs_blind, n_blind, refnames, refranges)

result, enes, ell, _ = fit(data_blind)
n_est = result.x

print('Initial   number of events :', *['{:6.2f}, '.format(ni) for ni in n_blind])
print('Estimated number of events :', *['{:6.2f}, '.format(ni) for ni in n_est])
# it also plots the histogram, the fit function, and the pdfs samples
pltfn.plot_fit_ell(enes, n_est, ell, parnames = sample_names_latex)

In [ ]:
nis, tmus = fn.tmu_scan(enes, n_est, ell, sizes = (3., 3.))
pltfn.plot_tmu_scan(nis, tmus, titles = sample_names_latex)

In [ ]:
cl    = 0.68
mucis = [efit.tmu_conf_int(ni, tmu, cl) for ni, tmu in zip(nis, tmus)]
for i, ci in enumerate(mucis):
    print('Number of {:s} events CI at {:4.0f} % CL = ({:5.2f}, {:5.2f})'.format(sample_names[i], 100*cl, *ci))

In [ ]:
for i, ci in enumerate(mucis):
    print('Number of {:s} events CI at {:4.0f} % CL = {:5.2f}  {:5.2f} +{:5.2f}'.format(sample_names[i], 100*cl, 
                                                                                        result.x[i],
                                                                                        *(ci - result.x[i])))

## Compute uncertainties on background event counts

In [ ]:
un_blind   = np.array([np.max(ci) - np.mean(ci) for ci in mucis])
effs_blind = np.array((collpars.eff_Bi_blind, collpars.eff_Tl_blind))

effs_E     = np.array((collpars.eff_Bi_E  , collpars.eff_Tl_E))
effs_RoI   = np.array((collpars.eff_Bi_RoI, collpars.eff_Tl_RoI))
